# TurboQuant VRAM Test

Run on Colab with a GPU runtime (A100 recommended for long context).
Compares baseline vs TurboQuant VRAM usage with the fused Triton attention kernel.

**Important:** TurboQuant saves KV cache memory. At short context (2K tokens),
the KV cache is only ~130 MB — invisible next to 16 GB of model weights.
You need **8K+ tokens** to see a meaningful VRAM difference. This notebook
defaults to 8K tokens where TurboQuant saves ~700+ MB of peak VRAM.

In [ ]:
!pip install -q turboquant transformers accelerate scipy

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"Triton available: {torch.cuda.is_available()}")
try:
    import triton
    print(f"Triton version: {triton.__version__}")
except ImportError:
    print("Triton not installed — will use PyTorch fallback attention")

In [ ]:
from turboquant import TurboQuantSession

MODEL = "meta-llama/Llama-3.1-8B-Instruct"

# 8K tokens — long enough that KV cache is a significant fraction of VRAM.
# At 2K tokens the KV cache is only ~130 MB (0.4% of total VRAM) — invisible.
# At 8K tokens the KV cache is ~1.1 GB — TurboQuant saves ~700-800 MB.
# Increase to 16K/32K for even more dramatic savings.
PROMPT = "The quick brown fox " * 2000  # ~8K tokens
BITS = 4
MAX_NEW = 32

In [ ]:
# --- Baseline ---
torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

session_bl = TurboQuantSession.from_pretrained(
    MODEL, variant="baseline", bits=BITS,
    dtype="auto", device_map="auto",
)
pre_bl = torch.cuda.memory_allocated()
text_bl = session_bl.generate(prompt=PROMPT, max_new_tokens=MAX_NEW)
peak_bl = torch.cuda.max_memory_allocated()
telem_bl = session_bl.last_telemetry()

print(f"Baseline peak: {peak_bl / 1e9:.2f} GB")
print(f"Baseline overhead: {(peak_bl - pre_bl) / 1e6:.0f} MB")
print(f"Baseline gen time: {telem_bl.get('generation_seconds', 'N/A')}s")
print(f"Output: {text_bl[:200]}...")

del session_bl
torch.cuda.empty_cache()

In [ ]:
# --- TurboQuant ---
torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

session_tq = TurboQuantSession.from_pretrained(
    MODEL, variant="qmse_packed", bits=BITS,
    dtype="auto", device_map="auto",
)
pre_tq = torch.cuda.memory_allocated()
text_tq = session_tq.generate(prompt=PROMPT, max_new_tokens=MAX_NEW)
peak_tq = torch.cuda.max_memory_allocated()
telem_tq = session_tq.last_telemetry()

print(f"TurboQuant peak: {peak_tq / 1e9:.2f} GB")
print(f"TurboQuant overhead: {(peak_tq - pre_tq) / 1e6:.0f} MB")
print(f"TurboQuant gen time: {telem_tq.get('generation_seconds', 'N/A')}s")
print(f"Packed KV: {telem_tq.get('packed_actual_bytes', 0) / 1e6:.0f} MB")
print(f"Dense KV (baseline): {telem_tq.get('dense_kv_bytes', 0) / 1e6:.0f} MB")
print(f"Payload savings: {telem_tq.get('payload_savings_percent', 0):.1f}%")
print(f"Output: {text_tq[:200]}...")

In [ ]:
# --- Comparison ---
overhead_bl = (peak_bl - pre_bl)
overhead_tq = (peak_tq - pre_tq)
savings_pct = (1 - overhead_tq / overhead_bl) * 100 if overhead_bl > 0 else 0
saved_mb = (overhead_bl - overhead_tq) / 1e6

print("=" * 60)
print(f"{'Metric':<30} {'Baseline':>14} {'TurboQuant':>14}")
print("-" * 60)
print(f"{'Peak VRAM (GB)':<30} {peak_bl/1e9:>14.2f} {peak_tq/1e9:>14.2f}")
print(f"{'KV overhead (MB)':<30} {overhead_bl/1e6:>14.0f} {overhead_tq/1e6:>14.0f}")
print(f"{'Gen time (s)':<30} {telem_bl.get('generation_seconds','?'):>14} {telem_tq.get('generation_seconds','?'):>14}")
print(f"{'VRAM saved (MB)':<30} {'':>14} {saved_mb:>14.0f}")
print(f"{'KV overhead savings %':<30} {'':>14} {savings_pct:>13.1f}%")
print(f"{'Output match':<30} {'':>14} {str(text_bl[:100] == text_tq[:100]):>14}")
print("=" * 60)
print()
if saved_mb < 100:
    print(f"NOTE: Only {saved_mb:.0f} MB saved. Try a longer prompt (16K-32K tokens)")
    print("to see more dramatic savings. KV cache scales linearly with context.")